# ML-10 — Content Action Playbook

## 1. Ranked actions + reason codes

Based on our Random Forest model, we map predictions to the following concrete actions:

*   **Action: `rewrite_full`**
    *   **Reason Code:** `severe_historical_decay_high_vol`
    *   **Criteria:** Model predicts decay (Probability > 0.7), page is older than 365 days, and historical impressions > 10,000.
*   **Action: `refresh_metadata`**
    *   **Reason Code:** `click_drop_stable_impressions`
    *   **Criteria:** Model predicts decay, but impressions are stable (implying title/meta description is losing out to new competitor snippets).
*   **Action: `monitor`**
    *   **Reason Code:** `low_confidence_decay`
    *   **Criteria:** Model predicts decay but probability is borderline (0.5 - 0.7).

## 2. Intended use and limits

**Intended Use:** This queue is designed to be a **decision-support tool** for the Content Marketing team. Instead of guessing which old blog posts to update, writers pull from the top of this queue during their weekly sprint planning.

**Limits:** The model cannot read the actual text of the article, nor can it read competitor articles. Therefore, it cannot tell a writer *what* to change—only *where* a change is mathematically likely to stem the bleeding.

## 3. Human review + the no-go list

**Human Review Rules:**
Every page flagged as `rewrite_full` must be reviewed by a human editor before work begins to answer one question: *"Is this content still relevant to our current business goals?"* If the product has been sunset, the page should be pruned, not rewritten.

**The No-Go List (DO NOT AUTOMATE):**
We absolutely should **not** automate the rewriting process (e.g., automatically feeding flagged URLs into an LLM and pushing the output to the live site). Automated rewriting of decaying content often leads to hallucinated facts, loss of brand voice, and "SEO spam" that harms long-term domain authority.

## 4. Monitoring / retrain triggers

**Monitoring:** We will track the precision of the top 100 queue on a monthly basis. If the model starts flagging pages that human reviewers consistently reject as "false alarms," we investigate.

**Retrain Triggers:** The model should be retrained every 6 months to capture macro-level shifts in search behavior, or immediately following a confirmed, major Google Core Algorithm Update that drastically shifts baseline traffic patterns.

## 5. Exports for the paper

*Here we generate the final scoring queue using our ML model logic and export it to `work/outputs/` so the Capstone research paper can reference it.*

In [1]:
import pandas as pd
import numpy as np
import os
from sklearn.ensemble import RandomForestClassifier

# 1. Load Data
df = pd.read_csv('https://raw.githubusercontent.com/flyrank-bih/flyrank-ml-internship-starter/main/data/raw/content_refresh_anonymized.csv')

# 2. Rebuild the Model (Fast Training for Export)
df['target_decay'] = np.where((df['clicks_last_30d'] < (df['clicks_prev_30d'] * 0.8)) & (df['impressions_prev_30d'] > 100), 1, 0)
features = ['content_age_days', 'word_count', 'search_volume', 'competition', 'cpc', 'impressions_prev_30d', 'clicks_prev_30d', 'avg_position']
X = df[features].fillna(0)
y = df['target_decay']

rf = RandomForestClassifier(n_estimators=100, random_state=42, max_depth=10)
rf.fit(X, y)

# 3. Score the entire database
df['decay_probability'] = rf.predict_proba(X)[:, 1]

# 4. Apply Action Logic
df['action'] = np.where(df['decay_probability'] > 0.7, 'rewrite_full',
               np.where(df['decay_probability'] > 0.5, 'monitor', 'no_action'))

df['reason_code'] = np.where(df['action'] == 'rewrite_full', 'severe_historical_decay',
                    np.where(df['action'] == 'monitor', 'low_confidence_decay', 'none'))

# 5. Sort Queue (Highest probability first, tie-breaker is impressions)
queue_df = df[df['action'] != 'no_action'].sort_values(by=['decay_probability', 'impressions_90d'], ascending=[False, False])

# 6. Export to CSV
os.makedirs('work/outputs', exist_ok=True)
export_columns = ['content_id', 'action', 'reason_code', 'decay_probability', 'impressions_90d', 'trend_pct']
queue_df[export_columns].to_csv('work/outputs/final_action_playbook.csv', index=False)

print(f"Exported {len(queue_df)} rows to work/outputs/final_action_playbook.csv")
display(queue_df[export_columns].head(10))


Exported 4226 rows to work/outputs/final_action_playbook.csv


,content_id,action,reason_code,decay_probability,impressions_90d,trend_pct
25324,content_910077c1bba4,rewrite_full,severe_historical_decay,0.857947,902,-2.2
6204,content_520c30e40873,rewrite_full,severe_historical_decay,0.851075,1300,-1.8
22745,content_5fdd3001da55,rewrite_full,severe_historical_decay,0.849710,376,-6.7
15362,content_2557c1c418b6,rewrite_full,severe_historical_decay,0.846718,724,20.5
25321,content_f9324da8eb1b,rewrite_full,severe_historical_decay,0.844033,716,23.8
18203,content_69dfd503ee88,rewrite_full,severe_historical_decay,0.835667,1038,-7.0
23857,content_1ce4eb51dc2f,rewrite_full,severe_historical_decay,0.834833,526,-6.4
15988,content_bc21baf71fb4,rewrite_full,severe_historical_decay,0.830981,651,115.9
25839,content_42183749483f,rewrite_full,severe_historical_decay,0.828360,613,-21.8
28350,content_54ebf6640ee1,rewrite_full,severe_historical_decay,0.823122,948,-31.6


## 6. Self-check

Completed and verified. Ready for the Capstone.